In [3]:
import requests
from bs4 import BeautifulSoup

# URL of the page to scrape
url = "https://theamazingworldofgumball.fandom.com/wiki/Category:Transcripts"

transcript_urls = []

while url:
    # Send a GET request to the URL
    response = requests.get(url)
    response.raise_for_status()  # Check if the request was successful

    # Parse the HTML content of the page
    soup = BeautifulSoup(response.content, "html.parser")

    # Find all links to transcript pages
    transcript_link_tags = soup.select("div.category-page__members a.category-page__member-link")

    # Extract the href attribute from each link and filter those ending with "/Transcript"
    transcript_urls.extend(["https://theamazingworldofgumball.fandom.com" + link["href"] for link in transcript_link_tags if link["href"].endswith("/Transcript")])

    # Find the "Next" button link
    next_button = soup.select_one("a.category-page__pagination-next")

    # Update the URL to the next page or set to None if no more pages
    url = next_button["href"] if next_button else None

In [4]:
def split_text_into_chunks(total_text, chunk_size=512):
    words = total_text.split()
    chunks = []
    current_chunk = ""

    for word in words:
        if len(current_chunk) + len(word) + 1 > chunk_size:
            chunks.append(current_chunk)
            current_chunk = word
        else:
            if current_chunk:
                current_chunk += " " + word
            else:
                current_chunk = word

    if current_chunk:
        chunks.append(current_chunk)

    return chunks

def sub_text_into_chunks(total_text, chunk_size=1024):
    words = total_text.split(' ')
    chunks = []
    current_chunk = ""

    for word in words:
        if len(current_chunk) + len(word) + 1 > chunk_size:
            chunks.append(current_chunk)
            current_chunk = ""
            break
        else:
            if current_chunk:
                current_chunk += " " + word
            else:
                current_chunk = word

    if current_chunk and len(chunks) == 0:
        chunks.append(current_chunk)

    return chunks

In [5]:
import re
import json
import unicodedata
from tqdm import tqdm
from bs4 import BeautifulSoup
import requests
from concurrent.futures import ThreadPoolExecutor

chunk_size = 1024
def process_transcript(transcript_url):
    """Fetch and process a single transcript URL."""
    try:
        transcript_response = requests.get(transcript_url)
        transcript_response.raise_for_status()

        soup = BeautifulSoup(transcript_response.content, "html.parser")
        transcript = soup.find("div", class_="mw-parser-output")
        title = soup.find("h1", class_="page-header__title").get_text(strip=True).replace("/Transcript", "")
        h2 = ""
        dl = ""
        chunks = []

        for child in transcript.children:
            if child.name == "h2":
                if h2 and dl:
                    text = f"Tell me a story about The Amazing World Of Gumball:\nTitle: {title}\nHeadline: {h2}\n"
                    dl = unicodedata.normalize("NFKD", dl)
                    chunked_dl = split_text_into_chunks(dl, chunk_size=chunk_size)
                    chunks.extend([text + chunk for chunk in chunked_dl])
                h2 = child.get_text(strip=True).replace("[]", "").replace("\u200B", " ").replace("]", "] ").replace("[", " [")
                dl = ""
            if child.name == "dl":
                content = ""
                for dd in child.find_all("dd", recursive=False):
                    dd_text = dd.get_text(strip=True).replace("[]", "").replace("\u200B", " ").replace("]", "] ").replace("[", " [")
                    dd_text = re.sub(r'\s+', ' ', dd_text)
                    content += dd_text + "\n"
                dl += content

        # Add remaining text
        if h2 and dl:
            text = f"Tell me a story about The Amazing World Of Gumball:\nTitle: {title}\nHeadline: {h2}\n{dl}"
            text = unicodedata.normalize("NFKD", text)
            chunks.extend(split_text_into_chunks(text, chunk_size=chunk_size))

        return chunks
    except Exception as e:
        print(f"Error processing URL {transcript_url}: {e}")
        return []

with open("transcripts.jsonl", "w", encoding="utf-8") as file, ThreadPoolExecutor(max_workers=16) as executor:
    # Submit tasks to the executor
    future_to_url = {executor.submit(process_transcript, url): url for url in transcript_urls}

    # Process results as they complete
    for future in tqdm(future_to_url, desc="Processing transcripts", total=len(transcript_urls)):
        chunks = future.result()
        for chunk in chunks:
            data = {"text": chunk}
            json.dump(data, file, ensure_ascii=False)
            file.write("\n")

Processing transcripts: 100%|██████████| 283/283 [00:54<00:00,  5.16it/s]


In [6]:
url = "https://theamazingworldofgumball.fandom.com/wiki/Category:Supporting_Characters"

character_urls = []

while url:
    # Send a GET request to the URL
    response = requests.get(url)
    response.raise_for_status()  # Check if the request was successful

    # Parse the HTML content of the page
    soup = BeautifulSoup(response.content, "html.parser")

    # Find all links to character pages
    character_link_tags = soup.select("li.category-page__member a.category-page__member-link")

    # Extract the href attribute from each link and filter those that have "Category" in the URL
    character_urls.extend(["https://theamazingworldofgumball.fandom.com" + link["href"] for link in character_link_tags if "Category" not in link["href"] and "Thread" not in link["href"]])
    
    # Find the "Next" button link
    next_button = soup.select_one("a.category-page__pagination-next")
    
    # Update the URL to the next page or set to None if no more pages
    url = next_button["href"] if next_button else None

character_urls += ["https://theamazingworldofgumball.fandom.com/wiki/Anais_Watterson", "https://theamazingworldofgumball.fandom.com/wiki/Darwin_Watterson", "https://theamazingworldofgumball.fandom.com/wiki/Gumball_Watterson", "https://theamazingworldofgumball.fandom.com/wiki/Nicole_Watterson", "https://theamazingworldofgumball.fandom.com/wiki/Richard_Watterson"]
character_urls += ["https://theamazingworldofgumball.fandom.com/wiki/Anais_Watterson", "https://theamazingworldofgumball.fandom.com/wiki/Darwin_Watterson", "https://theamazingworldofgumball.fandom.com/wiki/Gumball_Watterson", "https://theamazingworldofgumball.fandom.com/wiki/Nicole_Watterson", "https://theamazingworldofgumball.fandom.com/wiki/Richard_Watterson"]
character_urls += ["https://theamazingworldofgumball.fandom.com/wiki/Anais_Watterson", "https://theamazingworldofgumball.fandom.com/wiki/Darwin_Watterson", "https://theamazingworldofgumball.fandom.com/wiki/Gumball_Watterson", "https://theamazingworldofgumball.fandom.com/wiki/Nicole_Watterson", "https://theamazingworldofgumball.fandom.com/wiki/Richard_Watterson"]
character_urls += ["https://theamazingworldofgumball.fandom.com/wiki/Anais_Watterson", "https://theamazingworldofgumball.fandom.com/wiki/Darwin_Watterson", "https://theamazingworldofgumball.fandom.com/wiki/Gumball_Watterson", "https://theamazingworldofgumball.fandom.com/wiki/Nicole_Watterson", "https://theamazingworldofgumball.fandom.com/wiki/Richard_Watterson"]
character_urls += ["https://theamazingworldofgumball.fandom.com/wiki/Anais_Watterson", "https://theamazingworldofgumball.fandom.com/wiki/Darwin_Watterson", "https://theamazingworldofgumball.fandom.com/wiki/Gumball_Watterson", "https://theamazingworldofgumball.fandom.com/wiki/Nicole_Watterson", "https://theamazingworldofgumball.fandom.com/wiki/Richard_Watterson"]
character_urls += ["https://theamazingworldofgumball.fandom.com/wiki/Anais_Watterson", "https://theamazingworldofgumball.fandom.com/wiki/Darwin_Watterson", "https://theamazingworldofgumball.fandom.com/wiki/Gumball_Watterson", "https://theamazingworldofgumball.fandom.com/wiki/Nicole_Watterson", "https://theamazingworldofgumball.fandom.com/wiki/Richard_Watterson"]

In [7]:
import json
from tqdm import tqdm
from bs4 import BeautifulSoup
import requests
from concurrent.futures import ThreadPoolExecutor

def process_character(character_url):
    """Fetch and process a single character URL."""
    try:
        character_response = requests.get(character_url)
        character_response.raise_for_status()

        soup = BeautifulSoup(character_response.content, "html.parser")
        title = soup.find("h1", class_="page-header__title").get_text(strip=True)
        span = soup.find("span", class_="mw-headline", string="Personality")
        if not span:
            return "", ""
        about = span.find_next("p").get_text(strip=True)
        
    except Exception as e:
        print(f"Error processing URL {character_url}: {e}")
        return "", ""
    
    return title, about
    
with open("characters.jsonl", "w", encoding="utf-8") as file, ThreadPoolExecutor(max_workers=16) as executor:
    # Submit tasks to the executor
    future_to_url = {executor.submit(process_character, url): url for url in character_urls}

    # Process results as they complete
    for future in tqdm(future_to_url, desc="Processing characters", total=len(character_urls)):
        title, about = future.result()
        if title and about:
            data = {"text": f"Character: {title}\nAbout: {about}"}
            json.dump(data, file, ensure_ascii=False)
            file.write("\n")

Processing characters: 100%|██████████| 72/72 [00:09<00:00,  7.24it/s]


In [8]:
# combine the two jsonl files into one
import json

with open("transcripts.jsonl", "r", encoding="utf-8") as file:
    transcripts = [json.loads(line) for line in file]
    
with open("characters.jsonl", "r", encoding="utf-8") as file:
    characters = [json.loads(line) for line in file]
    
with open(f"gumball_{chunk_size}.jsonl", "w", encoding="utf-8") as file:
    for transcript in transcripts:
        json.dump(transcript, file, ensure_ascii=False)
        file.write("\n")
        
    for character in characters:
        json.dump(character, file, ensure_ascii=False)
        file.write("\n")